* Make sure to first run the ingest pipeline to simulate already existing data

In [ ]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.runner import execute_query
# import update
from helpers import *
from metadata import *
from silver_job import silver_job
from bronze_job import bronze_job

* Run an update pipeline with no monitoring enabled (make sure to remove any monitoring code that cannot be controlled by the `write_metadata=False` parameter - we want to do that in order to not execute useless `.count()` calls that will not be recorded in the metadata)

* (`taxi_trips_v2.json`, `air_quality_v2.json`,  `weather_v2.json`)

In [ ]:
spark = (
    SparkSession.builder
    .appName(f"MonitorOverheadOfUpdate")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.maxPlanStringLength", "8192")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

run_context = create_run_context()

config_paths = [
    "/opt/spark/configs/taxi_trips_v2.json",
    "/opt/spark/configs/air_quality_v2.json",
    "/opt/spark/configs/weather_v2.json"
]

try:
    bronze_times = {}
    for config_path in config_paths:
        config = load_config(config_path)
        dataset = config["dataset"]
        print(f"Loading dataset: {dataset}")

        bronze_times[dataset] = bronze_job(spark, config, run_context, mode="append", write_metadata=False)

    silver_times = {}
    for config_path in config_paths:
        config = load_config(config_path)

        silver_times[config["dataset"]] = silver_job(spark, config, run_context, incremental=True, write_metadata=False)

    print(f"BRONZE= {bronze_times}")

    print(f"SILVER= {silver_times}")

finally:
    spark.stop()

* Compare results with the results of the `incremental_update_time` notebook (that has monitoring enabled)